# Dynamic Portfolio Optimization via MDP & Reinforcement Learning

This notebook demonstrates the end-to-end quantitative portfolio allocation pipeline:
1. **Multi-Asset Covariance & Expected Return Modeling** across regime states (Bull, Neutral, Bear).
2. **Continuous Mean-Variance QP with CVaR and Transaction Cost Constraints**.
3. **Discrete MDP Value Iteration** with full Markov transition matrix P(s' | s).
4. **Model-Free Q-Learning** with experience replay.
5. **Multi-Period Backtesting & Performance Attribution** (Sharpe Ratio, Max Drawdown, 95% CVaR).

In [1]:
import numpy as np
import pandas as pd
from portfolio_mdp import PortfolioMDPOptimizer
from q_learning import QLearningPortfolioAgent
from backtest import PortfolioBacktester

In [2]:
# 1. Define Multi-Asset Universe
assets = ['US_Equities', 'Intl_Equities', 'Treasury_Bonds', 'Commodities']
mu_base = np.array([0.12, 0.10, 0.04, 0.07])
cov_base = np.array([
    [0.035, 0.025, 0.001, 0.015],
    [0.025, 0.040, 0.000, 0.018],
    [0.001, 0.000, 0.005, -0.002],
    [0.015, 0.018, -0.002, 0.045]
])

# Initialize Optimizer with 20 bps turnover cost and CVaR penalty
opt = PortfolioMDPOptimizer(
    asset_names=assets,
    expected_returns=mu_base,
    covariance_matrix=cov_base,
    risk_aversion=2.5,
    cvar_lambda=1.5,
    cvar_alpha=0.95,
    transaction_fee=0.002
)

print('Markov Transition Matrix P(s' | s):')
print(pd.DataFrame(opt.P, index=opt.regimes, columns=opt.regimes))

In [3]:
# 2. Continuous Mean-Variance + CVaR Optimization per Regime
for r_idx, r_name in enumerate(opt.regimes):
    w_opt = opt.optimize_weights_mean_variance(regime_idx=r_idx)
    mu_s, cov_s = opt.get_regime_moments(r_idx)
    var_95, cvar_95 = opt.calculate_cvar_parametric(w_opt, mu_s, cov_s)
    print(f'=== Regime: {r_name} ===')
    for a, w in zip(assets, w_opt):
        print(f'  {a:>15}: {w*100:.1f}%')
    print(f'  95% Analytical CVaR: {cvar_95*100:.2f}%')

In [4]:
# 3. MDP Value Iteration over Discrete Simplex Actions
actions = opt.generate_action_grid(step=0.25)
policy, V, _ = opt.value_iteration_policy(actions, discount=0.95)
print(f'Solved MDP Value Iteration for {len(actions)} allocation states.')
print(f'Value Function V(s, w) shape: {V.shape}')

In [5]:
# 4. Train Model-Free Q-Learning Agent
agent = QLearningPortfolioAgent(opt, action_grid=actions, alpha=0.15, gamma=0.95, epsilon=0.8)
history = agent.train(num_episodes=200, episode_length=40)
q_policy = agent.get_policy()
print('Q-Learning training complete. Final epsilon:', round(agent.epsilon, 4))

In [6]:
# 5. Multi-Period Backtesting Comparison
backtester = PortfolioBacktester(opt, initial_capital=100000.0)
regime_path = backtester.simulate_regime_path(num_periods=104, seed=42)

def mdp_eval_policy(s, w_curr, t):
    w_idx = np.argmin([np.linalg.norm(w_curr - a) for a in actions])
    return actions[policy[s, w_idx]]

def equal_weight_policy(s, w_curr, t):
    return np.full(opt.N, 1.0 / opt.N)

res_mdp = backtester.run_backtest(mdp_eval_policy, regime_path, seed=101)
res_eq = backtester.run_backtest(equal_weight_policy, regime_path, seed=101)

summary = pd.DataFrame([
    {
        'Strategy': 'Dynamic MDP Policy',
        'Total Return (%)': f"{res_mdp['total_return']*100:.2f}%",
        'Sharpe Ratio': f"{res_mdp['sharpe_ratio']:.2f}",
        'Max Drawdown (%)': f"{res_mdp['max_drawdown']*100:.2f}%",
        '95% CVaR (%)': f"{res_mdp['cvar_95']*100:.2f}%",
        'Avg Turnover': f"{res_mdp['avg_turnover']*100:.1f}%"
    },
    {
        'Strategy': 'Equal Weight (1/N)',
        'Total Return (%)': f"{res_eq['total_return']*100:.2f}%",
        'Sharpe Ratio': f"{res_eq['sharpe_ratio']:.2f}",
        'Max Drawdown (%)': f"{res_eq['max_drawdown']*100:.2f}%",
        '95% CVaR (%)': f"{res_eq['cvar_95']*100:.2f}%",
        'Avg Turnover': f"{res_eq['avg_turnover']*100:.1f}%"
    }
])
print(summary)